# Cross-Sectional Return Autoencoder Trading

This notebook tests a cross-sectional embedding idea:

1. Build one daily return vector across the whole equity universe. If the universe has 100 symbols, each date is a 100-dimensional vector.
2. Train an undercomplete autoencoder on pre-2020 daily cross-sectional return vectors.
3. Score out-of-sample dates by signed reconstruction residual per symbol.
4. Trade next session from the residual: positive signed squared residuals are long candidates, negative signed squared residuals are short candidates.

The signal is intentionally cross-sectional. The model does not look at one symbol in isolation; it learns the usual daily return manifold of the full universe and highlights symbol-level deviations from that manifold.

Lookahead control: reconstruction residuals use close-to-close returns known after the signal date closes. Portfolio weights are shifted to the next trading session before backtesting.

In [1]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from time import perf_counter
import sys

def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'quant_orchestrator').is_dir() and (candidate / 'pyproject.toml').exists():
            return candidate
    raise RuntimeError('Could not locate quant-orchestrator repo root')

REPO_ROOT = find_repo_root(Path.cwd().resolve())
for candidate in reversed((REPO_ROOT, REPO_ROOT.parent / 'quant-warehouse')):
    candidate_str = str(candidate)
    if candidate.exists():
        sys.path[:] = [entry for entry in sys.path if entry != candidate_str]
        sys.path.insert(0, candidate_str)
for module_name in list(sys.modules):
    if module_name == 'quant_orchestrator' or module_name.startswith('quant_orchestrator.'):
        sys.modules.pop(module_name, None)
    if module_name == 'quant_warehouse' or module_name.startswith('quant_warehouse.'):
        sys.modules.pop(module_name, None)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from IPython.display import display, Markdown

from quant_orchestrator.platforms.backtesting_frameworks.zipline.shared_book import (
    ZiplineSharedBookSummaryJob,
    run_zipline_shared_book_summary_jobs,
)
from quant_orchestrator.platforms.backtesting_frameworks.shared_book import (
    run_shared_book_backtest,
    shared_book_performance_metrics,
)
from quant_orchestrator.platforms.ml_frameworks.torch.runtime import configure_torch_runtime
from quant_warehouse.research_tools import FamilyEvaluationConfig, screen_fmp_equity_universe
from quant_warehouse.warehouse.api import Warehouse

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 180)


In [2]:
# Experiment controls
PROVIDER = 'fmp'
MIN_MARKET_CAP = 10_000_000_000
START_DATE = '2000-01-01'
END_DATE = None
TRAIN_END = pd.Timestamp('2019-12-31')
OOS_START = pd.Timestamp('2020-01-01')

CAPITAL_BASE = 1_000_000.0
COMMISSION_PER_SHARE = 0.005
SLIPPAGE_BPS = 5.0
ZIPLINE_MAX_WORKERS = 1
ZIPLINE_SYMBOL_LIMIT = 150

AE_EPOCHS = 80
AE_BATCH_SIZE = 512
AE_LR = 1e-3
AE_WEIGHT_DECAY = 1e-4
DENOISE_STD = 0.01
LATENT_DIM_RATIO = 0.20
MIN_LATENT_DIM = 2
MAX_LATENT_DIM = 32
HIDDEN_MULTIPLIER = 1.5
RANDOM_SEED = 7
REQUIRE_CUDA = False

TOP_K_VALUES = [5, 10, 20, 40]
ENTRY_QUANTILES = [0.90, 0.95, 0.975]
VARIANTS = ['long_only', 'short_only', 'long_short']

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)


## Load Universe And Prices

For the first iteration this uses the 1T+ FMP equity universe, matching the fast ML trading smoke tests. Scaling to 100B+ should work mechanically, but the AE input dimension grows with the number of symbols.

In [3]:
warehouse = Warehouse()
feature_config = FamilyEvaluationConfig(
    provider=PROVIDER,
    market_cap_min=MIN_MARKET_CAP,
    start_date=START_DATE,
    end_date=END_DATE,
)

symbols, raw_universe, universe_eligibility, universe_source = screen_fmp_equity_universe(
    feature_config,
    warehouse=warehouse,
)
symbols = tuple(sorted(str(symbol).upper() for symbol in symbols))
print({'symbols': len(symbols), 'universe_source': universe_source})
display(universe_eligibility.head(20))


{'symbols': 740, 'universe_source': 'openbb:fmp'}


,symbol,eligible,reason,screen_market_cap
0,NVDA,True,ok,4785585180000
1,GOOGL,True,ok,4368788098535
2,GOOG,True,ok,4349493623926
3,AAPL,True,ok,4323663859280
4,MSFT,True,ok,2854597080400
5,AMZN,True,ok,2599991070000
6,SPCX,True,ok,2059971841733
7,AVGO,True,ok,1757164597200
8,TSLA,True,ok,1597307716000
9,META,True,ok,1555825021126


In [4]:
def load_price_frames(symbols: tuple[str, ...]) -> dict[str, pd.DataFrame]:
    frames: dict[str, pd.DataFrame] = {}
    for symbol in symbols:
        prices = warehouse.read_prices(symbol, provider=PROVIDER, start=START_DATE, end=END_DATE)
        if prices is None or prices.empty:
            continue
        frame = prices.rename(columns=str.lower).copy()
        required = ['open', 'high', 'low', 'close', 'volume']
        if not set(required).issubset(frame.columns):
            continue
        frame = frame[required].apply(pd.to_numeric, errors='coerce')
        frame.index = pd.DatetimeIndex(pd.to_datetime(frame.index, errors='coerce')).normalize()
        frame = frame.dropna(subset=['open', 'high', 'low', 'close']).sort_index()
        if not frame.empty:
            frames[symbol] = frame
    return frames

price_frames = load_price_frames(symbols)
wide_close = pd.DataFrame({symbol: frame['close'] for symbol, frame in price_frames.items()}).sort_index().ffill()
wide_close = wide_close.dropna(axis=1, thresh=max(252, int(len(wide_close) * 0.30)))
wide_returns = wide_close.pct_change().replace([np.inf, -np.inf], np.nan)
wide_returns = wide_returns.dropna(how='all')

train_returns = wide_returns.loc[wide_returns.index <= TRAIN_END].dropna(axis=1, thresh=252)
oos_returns = wide_returns.loc[wide_returns.index >= OOS_START, train_returns.columns]
common_symbols = tuple(train_returns.columns.intersection(oos_returns.columns))
wide_returns = wide_returns.loc[:, list(common_symbols)]
train_returns = train_returns.loc[:, list(common_symbols)]
oos_returns = oos_returns.loc[:, list(common_symbols)]

print({
    'price_symbols': len(price_frames),
    'model_symbols': len(common_symbols),
    'return_dates': len(wide_returns),
    'train_dates': len(train_returns),
    'oos_dates': len(oos_returns),
    'start': str(wide_returns.index.min().date()),
    'end': str(wide_returns.index.max().date()),
})
display(wide_returns.describe().T[['mean', 'std', 'min', 'max']].head(20))


{'price_symbols': 740, 'model_symbols': 631, 'return_dates': 6657, 'train_dates': 5030, 'oos_dates': 1627, 'start': '2000-01-04', 'end': '2026-06-24'}


,mean,std,min,max
A,0.000504,0.025617,-0.349486,0.490157
AA,0.000429,0.030247,-0.210677,0.283333
AAL,0.000803,0.040653,-0.303672,0.586614
AAOI,0.002467,0.058040,-0.340749,0.670713
AAPL,0.001183,0.024149,-0.518692,0.153286
ABBV,0.000856,0.016627,-0.162616,0.137628
ABT,0.000467,0.015094,-0.160578,0.124281
ACN,0.000585,0.019016,-0.179668,0.163424
ADBE,0.000727,0.026505,-0.297569,0.239879
ADI,0.000732,0.025601,-0.166128,0.229344


## Train Cross-Sectional Autoencoder

Rows are dates and columns are symbols. The AE learns to reconstruct the whole cross-section of daily returns. The latent dimension is bounded so the model has to compress the cross-section instead of memorizing every symbol independently.

In [5]:
def robust_standardize_fit(frame: pd.DataFrame):
    raw = frame.to_numpy(dtype='float64', copy=True)
    lower = np.nanpercentile(raw, 0.1, axis=0)
    upper = np.nanpercentile(raw, 99.9, axis=0)
    clipped = np.clip(raw, lower, upper)
    center = np.nanmedian(clipped, axis=0)
    q1 = np.nanpercentile(clipped, 25.0, axis=0)
    q3 = np.nanpercentile(clipped, 75.0, axis=0)
    scale = q3 - q1
    center = np.nan_to_num(center, nan=0.0, posinf=0.0, neginf=0.0)
    scale = np.where(np.isfinite(scale) & (scale > 1e-9), scale, 1.0)
    lower = np.nan_to_num(lower, nan=-np.inf, posinf=np.inf, neginf=-np.inf)
    upper = np.nan_to_num(upper, nan=np.inf, posinf=np.inf, neginf=-np.inf)
    filled = np.where(np.isfinite(clipped), clipped, center)
    x = (filled - center) / scale
    return np.clip(np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0), -50, 50).astype('float32'), center, scale, lower, upper


def robust_standardize_apply(frame: pd.DataFrame, center, scale, lower, upper):
    raw = frame.to_numpy(dtype='float64', copy=True)
    clipped = np.clip(raw, lower, upper)
    filled = np.where(np.isfinite(clipped), clipped, center)
    x = (filled - center) / scale
    return np.clip(np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0), -50, 50).astype('float32')


class CrossSectionalReturnAutoEncoder(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int, latent_dim: int) -> None:
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, in_dim),
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))


def train_autoencoder(x_train: np.ndarray):
    runtime = configure_torch_runtime(require_cuda=REQUIRE_CUDA)
    device = runtime.torch_device
    in_dim = x_train.shape[1]
    hidden_dim = int(max(4, min(512, round(in_dim * HIDDEN_MULTIPLIER))))
    latent_dim = int(max(MIN_LATENT_DIM, min(MAX_LATENT_DIM, round(in_dim * LATENT_DIM_RATIO), max(1, in_dim - 1))))
    model = CrossSectionalReturnAutoEncoder(in_dim, hidden_dim, latent_dim).to(device)
    loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(torch.tensor(x_train, dtype=torch.float32)),
        batch_size=AE_BATCH_SIZE,
        shuffle=True,
    )
    optimizer = torch.optim.AdamW(model.parameters(), lr=AE_LR, weight_decay=AE_WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=AE_EPOCHS)
    loss_fn = nn.MSELoss()
    losses = []
    started = perf_counter()
    model.train()
    for epoch in range(AE_EPOCHS):
        epoch_losses = []
        for (batch,) in loader:
            batch = batch.to(device)
            noisy = batch + torch.randn_like(batch) * DENOISE_STD if DENOISE_STD > 0 else batch
            optimizer.zero_grad(set_to_none=True)
            recon = model(noisy)
            loss = loss_fn(recon, batch)
            if torch.isfinite(loss):
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                epoch_losses.append(float(loss.detach().cpu()))
        scheduler.step()
        losses.append(float(np.mean(epoch_losses)) if epoch_losses else np.nan)
    return model, {'device': device, 'in_dim': in_dim, 'hidden_dim': hidden_dim, 'latent_dim': latent_dim, 'fit_seconds': perf_counter() - started, 'losses': losses}


def reconstruct(model, x: np.ndarray, *, device: str, batch_size: int = 8192) -> np.ndarray:
    out = []
    model.eval()
    with torch.no_grad():
        for start in range(0, len(x), batch_size):
            end = min(start + batch_size, len(x))
            batch = torch.tensor(x[start:end], dtype=torch.float32, device=device)
            out.append(model(batch).detach().cpu().numpy().astype('float32'))
    return np.vstack(out) if out else np.empty_like(x)


In [6]:
x_train, center, scale, lower, upper = robust_standardize_fit(train_returns)
x_all = robust_standardize_apply(wide_returns, center, scale, lower, upper)

model, ae_info = train_autoencoder(x_train)
recon_all = reconstruct(model, x_all, device=ae_info['device'])

recon_returns = pd.DataFrame((recon_all * scale) + center, index=wide_returns.index, columns=wide_returns.columns)
residual = wide_returns - recon_returns
signed_squared_error = np.sign(residual) * (residual.abs() ** 2)
abs_squared_error = residual ** 2

ae_summary = {
    key: value for key, value in ae_info.items() if key != 'losses'
}
ae_summary.update({
    'final_loss': float(pd.Series(ae_info['losses']).dropna().iloc[-1]),
    'train_recon_mse': float(abs_squared_error.loc[abs_squared_error.index <= TRAIN_END].stack().mean()),
    'oos_recon_mse': float(abs_squared_error.loc[abs_squared_error.index >= OOS_START].stack().mean()),
})
print(ae_summary)
display(pd.Series(ae_info['losses'], name='train_loss').tail(10).to_frame())


{'device': 'cuda:0', 'in_dim': 631, 'hidden_dim': 512, 'latent_dim': 32, 'fit_seconds': 2.946626629214734, 'final_loss': 0.49626574516296384, 'train_recon_mse': 0.00036473048885739067, 'oos_recon_mse': 0.0005918473541313977}


,train_loss
70,0.495703
71,0.496104
72,0.496075
73,0.496203
74,0.495719
75,0.496668
76,0.496380
77,0.495997
78,0.495722
79,0.496266


## Build Residual Signals

Signal interpretation:

- `signed_squared_error > 0`: the symbol return was above the reconstructed cross-sectional manifold for that day.
- `signed_squared_error < 0`: the symbol return was below the reconstructed cross-sectional manifold for that day.

Weights are built on the signal date and shifted one trading session forward for execution.

In [7]:
def build_residual_weights(
    signed_error: pd.DataFrame,
    *,
    top_k: int,
    variant: str,
    entry_quantile: float,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    oos_error = signed_error.loc[signed_error.index >= OOS_START].copy()
    positive_threshold = oos_error.where(oos_error > 0).stack().quantile(entry_quantile)
    negative_threshold = oos_error.where(oos_error < 0).stack().quantile(1.0 - entry_quantile)
    weights = pd.DataFrame(0.0, index=oos_error.index, columns=oos_error.columns)
    rows = []
    for date, row in oos_error.iterrows():
        clean = row.dropna()
        longs = clean[clean >= positive_threshold].sort_values(ascending=False).head(top_k)
        shorts = clean[clean <= negative_threshold].sort_values(ascending=True).head(top_k)
        if variant == 'long_only':
            picks = longs
            if len(picks):
                weights.loc[date, picks.index] = 1.0 / float(top_k)
        elif variant == 'short_only':
            picks = shorts
            if len(picks):
                weights.loc[date, picks.index] = -1.0 / float(top_k)
        elif variant == 'long_short':
            if len(longs):
                weights.loc[date, longs.index] = 0.5 / float(top_k)
            if len(shorts):
                weights.loc[date, shorts.index] = -0.5 / float(top_k)
            picks = pd.concat([longs, shorts])
        else:
            raise ValueError(f'Unknown variant: {variant}')
        for symbol, score in picks.items():
            rows.append({
                'signal_date': date,
                'symbol': symbol,
                'score': float(score),
                'side': 'long' if score > 0 else 'short',
                'variant': variant,
                'top_k': top_k,
                'entry_quantile': entry_quantile,
            })
    shifted = weights.shift(1).fillna(0.0)
    return shifted, pd.DataFrame(rows)

signal_distribution = signed_squared_error.loc[signed_squared_error.index >= OOS_START].stack().describe(percentiles=[0.01, 0.025, 0.05, 0.5, 0.95, 0.975, 0.99])
display(signal_distribution.to_frame('signed_squared_error'))


,signed_squared_error
count,1.026637e+06
mean,2.135252e-04
std,1.496791e-01
min,-7.662823e-01
1%,-2.853602e-03
2.5%,-1.256519e-03
5%,-6.398286e-04
50%,-3.014571e-09
95%,6.648699e-04
97.5%,1.354078e-03


## Backtest Shared-Book Strategies

For smaller universes this attempts the native Zipline shared-book engine with realistic commission and slippage. For wider universes, the notebook uses the shared-book vectorized accounting path with the same shifted target weights and a Zipline-like cost model. That keeps the experiment focused on whether the cross-sectional residual signal works instead of failing inside Zipline process/memory overhead.


In [8]:
price_window_start = pd.Timestamp(OOS_START) - pd.Timedelta(days=20)
price_window_end = pd.Timestamp(wide_returns.index.max())
price_frame_subset = {
    symbol: frame.loc[
        (pd.DatetimeIndex(frame.index) >= price_window_start)
        & (pd.DatetimeIndex(frame.index) <= price_window_end)
    ].copy()
    for symbol, frame in price_frames.items()
    if symbol in common_symbols
}

def run_vectorized_residual_backtests(weight_records: list[dict[str, object]]) -> pd.DataFrame:
    rows = []
    # weights are already shifted one session forward, so align them to realized same-date returns.
    oos_returns = wide_returns.loc[wide_returns.index >= OOS_START, list(common_symbols)]
    cost_bps = SLIPPAGE_BPS + 0.5
    for record in weight_records:
        weights = record['weights']
        trades = record['trades']
        metadata = record['metadata']
        returns, equity, _turnover = run_shared_book_backtest(
            weights,
            oos_returns,
            cost_bps=cost_bps,
            capital_base=CAPITAL_BASE,
        )
        row = shared_book_performance_metrics(
            returns,
            equity,
            weights,
            trades,
            framework='vectorized_shared_book',
            variant=metadata['variant'],
            top_k=metadata['top_k'],
            cost_bps=cost_bps,
        )
        row.update(metadata)
        row['backtest_engine'] = 'vectorized_shared_book'
        rows.append(row)
    return pd.DataFrame(rows)

jobs = []
weight_records = []
trade_logs = []
for entry_quantile in ENTRY_QUANTILES:
    for variant in VARIANTS:
        for top_k in TOP_K_VALUES:
            weights, trades = build_residual_weights(
                signed_squared_error,
                top_k=top_k,
                variant=variant,
                entry_quantile=entry_quantile,
            )
            if weights.abs().sum(axis=1).sum() == 0:
                continue
            trades = trades.assign(strategy='cross_sectional_return_ae', variant=variant, top_k=top_k, entry_quantile=entry_quantile)
            trade_logs.append(trades)
            metadata = {
                'strategy': 'cross_sectional_return_ae',
                'variant': variant,
                'top_k': top_k,
                'entry_quantile': entry_quantile,
                'signal_events': len(trades),
                'symbols': len(common_symbols),
                'latent_dim': ae_summary['latent_dim'],
                'hidden_dim': ae_summary['hidden_dim'],
                'train_recon_mse': ae_summary['train_recon_mse'],
                'oos_recon_mse': ae_summary['oos_recon_mse'],
                'commission_per_share': COMMISSION_PER_SHARE,
                'slippage_bps': SLIPPAGE_BPS,
                'avg_gross_exposure': float(weights.abs().sum(axis=1).mean()),
                'avg_net_exposure': float(weights.sum(axis=1).mean()),
                'cash_days': float(weights.abs().sum(axis=1).eq(0).mean()),
            }
            weight_records.append({'weights': weights, 'trades': trades, 'metadata': metadata})
            if len(common_symbols) <= ZIPLINE_SYMBOL_LIMIT:
                jobs.append(
                    ZiplineSharedBookSummaryJob(
                        price_frames=price_frame_subset,
                        target_weights=weights,
                        metadata=metadata,
                        capital_base=CAPITAL_BASE,
                        commission_per_share=COMMISSION_PER_SHARE,
                        slippage_bps=SLIPPAGE_BPS,
                    )
                )

print({
    'candidate_jobs': len(weight_records),
    'zipline_jobs': len(jobs),
    'trade_rows': sum(len(frame) for frame in trade_logs),
    'symbols': len(common_symbols),
    'zipline_symbol_limit': ZIPLINE_SYMBOL_LIMIT,
})

if jobs:
    try:
        backtest_summary = run_zipline_shared_book_summary_jobs(jobs, max_workers=ZIPLINE_MAX_WORKERS)
        backtest_summary['backtest_engine'] = 'zipline_shared_book'
    except Exception as exc:
        print(f'Zipline failed ({type(exc).__name__}: {exc}); falling back to vectorized shared-book accounting.')
        backtest_summary = run_vectorized_residual_backtests(weight_records)
else:
    print('Using vectorized shared-book accounting because the universe is wider than ZIPLINE_SYMBOL_LIMIT.')
    backtest_summary = run_vectorized_residual_backtests(weight_records)

trade_log = pd.concat(trade_logs, ignore_index=True) if trade_logs else pd.DataFrame()
backtest_summary = backtest_summary.sort_values(['sharpe', 'total_return'], ascending=False).reset_index(drop=True)
display(backtest_summary.head(20))


{'candidate_jobs': 36, 'zipline_jobs': 0, 'trade_rows': 755296, 'symbols': 631, 'zipline_symbol_limit': 150}
Using vectorized shared-book accounting because the universe is wider than ZIPLINE_SYMBOL_LIMIT.


,framework,variant,top_k,cost_bps,days,trades,final_equity,total_return,annualized_return,annualized_vol,sharpe,max_drawdown,win_rate,avg_gross_exposure,avg_net_exposure,fully_invested_days,cash_days,strategy,entry_quantile,signal_events,symbols,latent_dim,hidden_dim,train_recon_mse,oos_recon_mse,commission_per_share,slippage_bps,backtest_engine
0,vectorized_shared_book,long_only,5,5.5,1627,7686,6.067781e+06,5.067781,0.322150,0.548103,0.767977,-0.533152,0.500307,0.944192,0.944192,0.867240,0.006146,cross_sectional_return_ae,0.950,7686,631,32,512,0.000365,0.000592,0.005,5.0,vectorized_shared_book
1,vectorized_shared_book,long_only,5,5.5,1627,8094,5.226300e+06,4.226300,0.291928,0.551560,0.724751,-0.559152,0.499693,0.994345,0.994345,0.989551,0.000615,cross_sectional_return_ae,0.900,8094,631,32,512,0.000365,0.000592,0.005,5.0,vectorized_shared_book
2,vectorized_shared_book,long_only,5,5.5,1627,6060,4.847458e+06,3.847458,0.276958,0.525762,0.711467,-0.487400,0.476337,0.744315,0.744315,0.509527,0.039336,cross_sectional_return_ae,0.975,6060,631,32,512,0.000365,0.000592,0.005,5.0,vectorized_shared_book
3,vectorized_shared_book,long_only,10,5.5,1627,8370,3.179457e+06,2.179457,0.196210,0.341845,0.690664,-0.394276,0.472034,0.513829,0.513829,0.183159,0.039336,cross_sectional_return_ae,0.975,8370,631,32,512,0.000365,0.000592,0.005,5.0,vectorized_shared_book
4,vectorized_shared_book,long_only,10,5.5,1627,13275,3.497822e+06,2.497822,0.214023,0.388199,0.690021,-0.460572,0.488015,0.815304,0.815304,0.556238,0.006146,cross_sectional_return_ae,0.950,13275,631,32,512,0.000365,0.000592,0.005,5.0,vectorized_shared_book
5,vectorized_shared_book,long_only,20,5.5,1627,9791,1.959524e+06,0.959524,0.109814,0.220858,0.581500,-0.358210,0.469576,0.300369,0.300369,0.046097,0.039336,cross_sectional_return_ae,0.975,9791,631,32,512,0.000365,0.000592,0.005,5.0,vectorized_shared_book
6,vectorized_shared_book,long_only,20,5.5,1627,18400,2.137513e+06,1.137513,0.124860,0.271800,0.568361,-0.408385,0.487400,0.564843,0.564843,0.188691,0.006146,cross_sectional_return_ae,0.950,18400,631,32,512,0.000365,0.000592,0.005,5.0,vectorized_shared_book
7,vectorized_shared_book,long_only,10,5.5,1627,15925,2.367106e+06,1.367106,0.142776,0.407098,0.527812,-0.557822,0.484942,0.978181,0.978181,0.936079,0.000615,cross_sectional_return_ae,0.900,15925,631,32,512,0.000365,0.000592,0.005,5.0,vectorized_shared_book
8,vectorized_shared_book,long_only,40,5.5,1627,21148,1.538823e+06,0.538823,0.069038,0.182183,0.457770,-0.357220,0.484327,0.324493,0.324493,0.043024,0.006146,cross_sectional_return_ae,0.950,21148,631,32,512,0.000365,0.000592,0.005,5.0,vectorized_shared_book
9,vectorized_shared_book,long_only,40,5.5,1627,10600,1.435405e+06,0.435405,0.057580,0.148387,0.451773,-0.308894,0.468961,0.162615,0.162615,0.015980,0.039336,cross_sectional_return_ae,0.975,10600,631,32,512,0.000365,0.000592,0.005,5.0,vectorized_shared_book


## Diagnostics

These diagnostics check whether the signed residual is behaving like a tradeable signal or just highlighting high-volatility names.

In [9]:
daily_signal_breadth = (signed_squared_error.loc[signed_squared_error.index >= OOS_START].abs() > 0).sum(axis=1)
residual_vol_corr = pd.DataFrame({
    'residual_abs_mean': signed_squared_error.loc[signed_squared_error.index >= OOS_START].abs().mean(axis=0),
    'realized_vol': wide_returns.loc[wide_returns.index >= OOS_START].std(axis=0),
}).dropna()
residual_vol_corr_value = residual_vol_corr['residual_abs_mean'].corr(residual_vol_corr['realized_vol'])

best_by_variant = (
    backtest_summary
    .sort_values(['variant', 'sharpe', 'total_return'], ascending=[True, False, False])
    .groupby('variant', as_index=False)
    .head(1)
    .reset_index(drop=True)
)

print({
    'daily_signal_breadth_mean': float(daily_signal_breadth.mean()),
    'daily_signal_breadth_min': int(daily_signal_breadth.min()),
    'daily_signal_breadth_max': int(daily_signal_breadth.max()),
    'residual_abs_vs_realized_vol_corr': float(residual_vol_corr_value),
})
display(best_by_variant)
display(residual_vol_corr.sort_values('residual_abs_mean', ascending=False).head(20))


{'daily_signal_breadth_mean': 631.0, 'daily_signal_breadth_min': 631, 'daily_signal_breadth_max': 631, 'residual_abs_vs_realized_vol_corr': 0.8429538767695026}


,framework,variant,top_k,cost_bps,days,trades,final_equity,total_return,annualized_return,annualized_vol,sharpe,max_drawdown,win_rate,avg_gross_exposure,avg_net_exposure,fully_invested_days,cash_days,strategy,entry_quantile,signal_events,symbols,latent_dim,hidden_dim,train_recon_mse,oos_recon_mse,commission_per_share,slippage_bps,backtest_engine
0,vectorized_shared_book,long_only,5,5.5,1627,7686,6.067781e+06,5.067781,0.322150,0.548103,0.767977,-0.533152,0.500307,0.944192,0.944192,0.867240,0.006146,cross_sectional_return_ae,0.950,7686,631,32,512,0.000365,0.000592,0.005,5.0,vectorized_shared_book
1,vectorized_shared_book,long_short,10,5.5,1627,16653,4.925828e+05,-0.507417,-0.103874,0.204830,-0.434197,-0.702042,0.463430,0.511278,0.002551,0.070682,0.003688,cross_sectional_return_ae,0.975,16653,631,32,512,0.000365,0.000592,0.005,5.0,vectorized_shared_book
2,vectorized_shared_book,short_only,40,5.5,1627,10658,3.204119e+05,-0.679588,-0.161620,0.185105,-0.859379,-0.771545,0.425937,0.163675,-0.163675,0.015980,0.047326,cross_sectional_return_ae,0.975,10658,631,32,512,0.000365,0.000592,0.005,5.0,vectorized_shared_book


,residual_abs_mean,realized_vol
MAIR,0.093593,0.305031
SMMT,0.011126,0.105676
GME,0.007490,0.089114
HUT,0.005669,0.075215
MDGL,0.005599,0.075266
CMSA,0.004423,0.066551
AAOI,0.004280,0.069896
CVNA,0.004160,0.068561
GSAT,0.003361,0.060675
DD,0.002618,0.053728


## Written Analysis

Run the next cell after the backtest. It generates a concise interpretation based on the actual output tables.

In [10]:
if backtest_summary.empty:
    display(Markdown('No backtest results were produced. Check price coverage, thresholds, and universe size.'))
else:
    best = backtest_summary.iloc[0]
    long_best = best_by_variant.loc[best_by_variant['variant'].eq('long_only')]
    short_best = best_by_variant.loc[best_by_variant['variant'].eq('short_only')]
    ls_best = best_by_variant.loc[best_by_variant['variant'].eq('long_short')]
    lines = [
        '### Analysis',
        '',
        f'- Universe: {len(common_symbols)} symbols; each date is a {len(common_symbols)}-dimensional return vector.',
        f'- Autoencoder: hidden_dim={ae_summary["hidden_dim"]}, latent_dim={ae_summary["latent_dim"]}, device={ae_summary["device"]}, fit_seconds={ae_summary["fit_seconds"]:.2f}.',
        f'- Reconstruction MSE: train={ae_summary["train_recon_mse"]:.8f}, OOS={ae_summary["oos_recon_mse"]:.8f}.',
        f'- Best strategy: variant={best["variant"]}, top_k={int(best["top_k"])}, entry_quantile={best["entry_quantile"]:.3f}, sharpe={best["sharpe"]:.3f}, total_return={best["total_return"]:.3f}, max_drawdown={best["max_drawdown"]:.3f}.',
        f'- Backtest engine: {best.get("backtest_engine", best.get("framework", "unknown"))}.',
        f'- Residual magnitude versus realized volatility correlation: {residual_vol_corr_value:.3f}. High values mean the signal may mostly be volatility exposure rather than cross-sectional structure.',
    ]
    for name, frame in [('Long-only', long_best), ('Short-only', short_best), ('Long-short', ls_best)]:
        if not frame.empty:
            row = frame.iloc[0]
            lines.append(f'- {name} best: sharpe={row["sharpe"]:.3f}, total_return={row["total_return"]:.3f}, max_drawdown={row["max_drawdown"]:.3f}, top_k={int(row["top_k"])}.')
    if float(best['sharpe']) > 1.0 and float(best['total_return']) > 0:
        lines.append('- Initial read: this is promising enough to scale beyond 1T and compare against classifier-only/AE feature-family strategies.')
    else:
        lines.append('- Initial read: this is not yet convincing; inspect whether residuals are dominated by volatility or a small number of symbols before scaling.')
    display(Markdown('\n'.join(lines)))


### Analysis

- Universe: 631 symbols; each date is a 631-dimensional return vector.
- Autoencoder: hidden_dim=512, latent_dim=32, device=cuda:0, fit_seconds=2.95.
- Reconstruction MSE: train=0.00036473, OOS=0.00059185.
- Best strategy: variant=long_only, top_k=5, entry_quantile=0.950, sharpe=0.768, total_return=5.068, max_drawdown=-0.533.
- Backtest engine: vectorized_shared_book.
- Residual magnitude versus realized volatility correlation: 0.843. High values mean the signal may mostly be volatility exposure rather than cross-sectional structure.
- Long-only best: sharpe=0.768, total_return=5.068, max_drawdown=-0.533, top_k=5.
- Short-only best: sharpe=-0.859, total_return=-0.680, max_drawdown=-0.772, top_k=40.
- Long-short best: sharpe=-0.434, total_return=-0.507, max_drawdown=-0.702, top_k=10.
- Initial read: this is not yet convincing; inspect whether residuals are dominated by volatility or a small number of symbols before scaling.